# Transformer from Scratch with Keras 3

```
Text --> [Tokenize] --> [Embed + Position] --> [Encoder] --> [Decoder] --> [Output]
```

Keras 3 is a multi-backend deep learning framework that runs on top of TensorFlow, JAX, and PyTorch.

> Make with: Antigravity AI coding assistant

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import ops, layers

# import os
# os.environ["KERAS_BACKEND"] = "jax"  # Or "torch" or "tensorflow"

keras.utils.set_random_seed(42)
print(f'Keras {keras.__version__}, backend: {keras.backend.backend()}')

---
## Step 1: Tokenization

Convert text to integer IDs using a vocabulary lookup.

In [ ]:
en_vocab = {'<PAD>': 0, '<UNK>': 1, 'the': 2, 'cat': 3, 'sat': 4, 'on': 5, 'mat': 6}
vn_vocab = {'<PAD>': 0, '<UNK>': 1, 'con': 2, 'mèo': 3, 'ngồi': 4, 'trên': 5, 'tấm': 6, 'thảm': 7}

en_sentence = 'the cat sat on the mat'
vn_sentence = 'con mèo ngồi trên tấm thảm'

en_words  = en_sentence.split()
vn_words  = vn_sentence.split()
en_tokens = [en_vocab.get(w, en_vocab['<UNK>']) for w in en_words]
vn_tokens = [vn_vocab.get(w, vn_vocab['<UNK>']) for w in vn_words]

print(f'EN: {en_words} --> {en_tokens}')
print(f'VN: {vn_words} --> {vn_tokens}')

---
## Step 2: Transformer Encoder (Keras 3)

- Token positions are encoded via a learnable **`layers.Embedding`** (built-in).
- Self-attention is handled by **`layers.MultiHeadAttention`** (built-in).

```
Input --> MultiHeadAttention --> Add & Norm --> FFN --> Add & Norm --> Output
```

In [ ]:
class EncoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.mha      = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)
        self.ffn      = keras.Sequential([layers.Dense(d_ff, activation='relu'), layers.Dense(d_model)])
        self.norm1    = layers.LayerNormalization(epsilon=1e-6)
        self.norm2    = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        attn = self.mha(x, x, x)
        x    = self.norm1(x + self.dropout1(attn, training=training))
        ffn  = self.ffn(x)
        return self.norm2(x + self.dropout2(ffn, training=training))


class TransformerEncoder(keras.Model):
    def __init__(self, num_layers, d_model, num_heads, d_ff, vocab_size, max_len, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model       = d_model
        self.embedding     = layers.Embedding(vocab_size, d_model)
        self.pos_embedding = layers.Embedding(max_len, d_model)   # learnable position encoding
        self.enc_layers    = [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        self.dropout       = layers.Dropout(dropout)

    def call(self, x, training=False):
        seq_len   = ops.shape(x)[1]
        positions = ops.arange(seq_len)
        x = self.embedding(x) * ops.sqrt(ops.cast(self.d_model, 'float32'))
        x = x + self.pos_embedding(positions)
        x = self.dropout(x, training=training)
        for layer in self.enc_layers:
            x = layer(x, training=training)
        return x

print('TransformerEncoder defined')

In [ ]:
D_MODEL, NUM_HEADS, D_FF, NUM_LAYERS, MAX_SEQ = 64, 4, 256, 2, 50

encoder    = TransformerEncoder(NUM_LAYERS, D_MODEL, NUM_HEADS, D_FF, len(en_vocab), MAX_SEQ)
enc_input  = np.array([en_tokens])
enc_output = encoder(enc_input, training=False)

print(f'Input:  {enc_input.shape}')
print(f'Output: {enc_output.shape}')
encoder.summary()

---
## Step 3: Transformer Decoder (Keras 3)

The Decoder adds two things over the Encoder:
1. **Causal mask** — `use_causal_mask=True` prevents attending to future tokens (built-in).
2. **Cross-attention** — Q from decoder, K/V from encoder output.

```
Target --> Masked Self-Attention --> Add & Norm
       --> Cross-Attention (Q=dec, K/V=enc) --> Add & Norm
       --> FFN --> Add & Norm --> Output
```

In [ ]:
class DecoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.masked_mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)
        self.cross_mha  = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)
        self.ffn        = keras.Sequential([layers.Dense(d_ff, activation='relu'), layers.Dense(d_model)])
        self.norm1      = layers.LayerNormalization(epsilon=1e-6)
        self.norm2      = layers.LayerNormalization(epsilon=1e-6)
        self.norm3      = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1   = layers.Dropout(dropout)
        self.dropout2   = layers.Dropout(dropout)
        self.dropout3   = layers.Dropout(dropout)

    def call(self, x, enc_output, training=False):
        attn1 = self.masked_mha(x, x, x, use_causal_mask=True)
        x     = self.norm1(x + self.dropout1(attn1, training=training))

        attn2 = self.cross_mha(x, enc_output, enc_output)
        x     = self.norm2(x + self.dropout2(attn2, training=training))

        ffn = self.ffn(x)
        return self.norm3(x + self.dropout3(ffn, training=training))


class TransformerDecoder(keras.Model):
    def __init__(self, num_layers, d_model, num_heads, d_ff, vocab_size, max_len, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model       = d_model
        self.embedding     = layers.Embedding(vocab_size, d_model)
        self.pos_embedding = layers.Embedding(max_len, d_model)   # learnable position encoding
        self.dec_layers    = [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        self.dropout       = layers.Dropout(dropout)

    def call(self, x, enc_output, training=False):
        seq_len   = ops.shape(x)[1]
        positions = ops.arange(seq_len)
        x = self.embedding(x) * ops.sqrt(ops.cast(self.d_model, 'float32'))
        x = x + self.pos_embedding(positions)
        x = self.dropout(x, training=training)
        for layer in self.dec_layers:
            x = layer(x, enc_output, training=training)
        return x

print('TransformerDecoder defined')

In [ ]:
decoder    = TransformerDecoder(NUM_LAYERS, D_MODEL, NUM_HEADS, D_FF, len(vn_vocab), MAX_SEQ)
dec_input  = np.array([vn_tokens])
dec_output = decoder(dec_input, enc_output, training=False)

print(f'Encoder output: {enc_output.shape}')
print(f'Decoder input:  {dec_input.shape}')
print(f'Decoder output: {dec_output.shape}')

---
## Step 4: Full Transformer + Training

Wrap Encoder + Decoder + output projection into one model.
Train with **teacher forcing** on a single EN→VN sentence pair.

```
Decoder input:  [<PAD>  con   mèo   ngồi  trên  tấm ]  (shifted right)
Labels:         [con    mèo   ngồi  trên  tấm   thảm]  (predict these)
```

In [ ]:
class Transformer(keras.Model):
    def __init__(self, num_layers, d_model, num_heads, d_ff,
                 src_vocab_size, tgt_vocab_size, max_len, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.encoder      = TransformerEncoder(num_layers, d_model, num_heads, d_ff, src_vocab_size, max_len, dropout)
        self.decoder      = TransformerDecoder(num_layers, d_model, num_heads, d_ff, tgt_vocab_size, max_len, dropout)
        self.output_layer = layers.Dense(tgt_vocab_size)

    def call(self, inputs, training=False):
        src, tgt = inputs
        enc_out  = self.encoder(src, training=training)
        dec_out  = self.decoder(tgt, enc_out, training=training)
        return self.output_layer(dec_out)


model = Transformer(
    NUM_LAYERS, D_MODEL, NUM_HEADS, D_FF,
    src_vocab_size=len(en_vocab),
    tgt_vocab_size=len(vn_vocab),
    max_len=MAX_SEQ
)

# Teacher forcing: decoder sees [<PAD>, con, mèo, ...] and predicts [con, mèo, ..., thảm]
src_train       = np.array([en_tokens])
tgt_train_input = np.array([[0] + vn_tokens[:-1]])  # shifted right
tgt_labels      = np.array([vn_tokens])

vn_vocab_list = list(vn_vocab.keys())
en_vocab_list = list(en_vocab.keys())

print('Teacher forcing:')
print(f'  Encoder input: {[en_vocab_list[i] for i in src_train[0]]}')
print(f'  Decoder input: {[vn_vocab_list[i] for i in tgt_train_input[0]]} (shifted right)')
print(f'  Labels:        {[vn_vocab_list[i] for i in tgt_labels[0]]}')

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy()]
)

history = model.fit(
    x=(src_train, tgt_train_input),
    y=tgt_labels,
    epochs=300,
    verbose=0
)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(history.history['loss'], color='#e74c3c', linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
plt.tight_layout()
plt.show()

In [ ]:
logits        = model((src_train, tgt_train_input), training=False)
probs         = keras.activations.softmax(logits, axis=-1)
predicted_ids = ops.argmax(probs[0], axis=-1)

print('Predictions after training:\n')
print(f'{"Pos":<5} {"Target":<8} {"Predicted":<10} {"Conf":<8}')
print('-' * 35)
for pos in range(len(vn_words)):
    target = vn_vocab_list[tgt_labels[0][pos]]
    pred   = vn_vocab_list[int(predicted_ids[pos])]
    conf   = float(probs[0][pos][int(predicted_ids[pos])])
    match  = 'ok' if pred == target else 'X'
    print(f'{pos:<5} {target:<8} {pred:<10} {conf:.3f}    {match}')

print(f'\n"{en_sentence}" --> "{" ".join([vn_vocab_list[int(i)] for i in predicted_ids])}"')

In [ ]:
model.summary()